In [ ]:
!pip install --no-index \
    -r /kaggle/input/notebooks/ttahara/birdclef-2026-download-wheels/requirements.txt \
    --find-links=/kaggle/input/notebooks/ttahara/birdclef-2026-download-wheels/wheels

In [ ]:
import gc
import copy
import random
import typing as tp
from pathlib import Path
from time import time
import numpy as np
import pickle
import pandas as pd
from tqdm.notebook import tqdm, trange
import soundfile
import timm
import torch
import torchaudio
from torchvision.transforms import v2 as tvt_v2
import torch.nn.functional as F
from torch import nn
from joblib import load as jb_load, Parallel, delayed
import threading
import openvino as ov
import io
import re
import os
import onnx

In [ ]:
ROOT = Path.cwd().parent
INPUT = ROOT / "input"
DATA = INPUT / "competitions" / "birdclef-2026"
TRAIN_AUDIO = DATA / "train_audio"
TRAIN_SS = DATA / "train_soundscapes"
TEST_SS = DATA / "test_soundscapes"
soundscape_labels = pd.read_csv(DATA / "train_soundscapes_labels.csv")
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
TRAINED_MODEL = Path("/kaggle/input/datasets/jakkma/tttttt")
N_CLASSES = 234
N_WINDOWS = 12
taxonomy = pd.read_csv(DATA / "taxonomy.csv")
CLASSES = taxonomy.primary_label.values.tolist()
label2idx = {label: idx for idx, label in enumerate(taxonomy.primary_label.values)}
idx2label = {idx: label for label, idx in label2idx.items()}

N_FOLDS = 1
RANK_AVG = False
USE_GAUSSIAN_SMOOTH = False
USE_PRIOR = True
USE_CLASS_TEMP = False
USE_FILE_SCALE = False
USE_RANK_SCALE = False
USE_ADAPT_SMOOTH = False

GAUSSIAN_KERNEL = np.array([0.25, 0.50, 0.25], dtype=np.float32)

IS_TEST_ENV = len(sample_sub) > 10
if IS_TEST_ENV:
    test_ss_paths = []
    added = set()
    for row_id in sample_sub["row_id"].values:
        file_id = "_".join(row_id.split("_")[:-1])
        if file_id in added:
            continue
        added.add(file_id)
        test_ss_paths.append(TEST_SS / f"{file_id}.ogg")
else:
    test_ss_paths = sorted(TRAIN_SS.iterdir())[:10]

test_ss_segs = []
for p in test_ss_paths:
    for i in range(0, 60, 5):
        test_ss_segs.append(f"{p.stem}_{i + 5}")
print(f"{test_ss_segs[0]}<-{len(test_ss_segs)}->{test_ss_segs[-1]}")

mel_spectrogram_params = dict(
    sample_rate=32_000,
    n_fft=2048,
    hop_length=512,
    f_min=0,
    f_max=16_000,
    n_mels=128,
    normalized=True,
)
lms_shape = (128, 313)
top_db = 80.0
print('Done')

In [ ]:
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m: return "unknown", -1
    _, site, _, hms = m.groups()
    return site, int(hms[:2])
    
def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    
    eps = 1e-4
    n = len(scores)
    out = scores.copy()

    p = np.tile(tables["global_p"], (n, 1))

    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]
            nh = tables["hour_n"][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]

    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]
            ns = tables["site_n"][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]

    p = np.clip(p, eps, 1 - eps)
    logit_prior = np.log(p) - np.log1p(-p)
    
    out += lambda_prior * logit_prior
    
    return out.astype(np.float32)

In [ ]:
# Class-Specific Temperatures
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
TEXTURE_TAXA   = {"Amphibia", "Insecta"}

temperatures = np.ones(N_CLASSES, dtype=np.float32)
for ci, label in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(label, "Aves")
    if cls in TEXTURE_TAXA:
        temperatures[ci] = 0.95
    else:
        temperatures[ci] = 1.10

n_texture = (temperatures < 1.0).sum()
n_event   = (temperatures > 1.0).sum()
print(f"Temperatures: {n_event} event species (T=1.10), {n_texture} texture species (T=0.95)")

In [ ]:
def file_confidence_scale(probs, n_windows=12, top_k=2, power=0.4):
    N, C = probs.shape
    assert N % n_windows == 0
    
    view      = probs.reshape(-1, n_windows, C)
    sorted_v  = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    
    scale  = np.power(top_k_mean, power)
    scaled = view * scale
    
    return scaled.reshape(N, C)

In [ ]:
def rank_aware_scaling(probs, n_windows=12, power=0.4):
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"

    view     = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)

    scale  = np.power(file_max, power)
    scaled = view * scale

    return scaled.reshape(N, C)

In [ ]:
def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    N, C = probs.shape
    assert N % n_windows == 0, f"Expected multiple of {n_windows}, got {N}"

    result = probs.copy()
    view   = probs.reshape(-1, n_windows, C)
    out    = result.reshape(-1, n_windows, C)

    for t in range(n_windows):

        conf = view[:, t, :].max(axis=-1, keepdims=True)

        alpha = base_alpha * (1.0 - conf)

        if t == 0:
            neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0

        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg

    return result

In [ ]:
def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, "bias") and layer.bias is not None:
        layer.bias.data.fill_(0.)

def init_bn(bn):
    bn.bias.data.fill_(0.)
    bn.weight.data.fill_(1.0)

class GeM1d(nn.Module):
    def __init__(self, p=3.0, kernel_size=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.kernel_size = kernel_size
        self.eps = eps

    def forward(self, x):
        x = x.clamp(min=self.eps)
        return F.avg_pool1d(
            x.pow(self.p), kernel_size=self.kernel_size,
            stride=1, padding=self.kernel_size // 2,
        ).pow(1.0 / self.p)

class AttBlock(nn.Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.att = nn.Conv1d(in_feat, out_feat, 1, bias=True)
        self.cla = nn.Conv1d(in_feat, out_feat, 1, bias=True)
        init_layer(self.att)
        init_layer(self.cla)

    def forward(self, x):
        norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
        framewise = self.cla(x)
        clipwise = torch.sum(norm_att * framewise, dim=2)
        return clipwise

class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
        init_layer(self.proj)

    def forward(self, feat_map):
        return self.proj(feat_map.mean(dim=[2, 3]))

class BirdModel(nn.Module):
    def __init__(self, model_name, pretrained=False, drop_path_rate=0.15, drop_rate=0.2,
                 num_classes=234, head_dropout=0.35, n_mels=128):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(n_mels)
        init_bn(self.bn0)

        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, in_chans=1,
            global_pool="", num_classes=0,
            drop_path_rate=drop_path_rate, drop_rate=drop_rate,
        )
        with torch.no_grad():
            n_feat = self.backbone(torch.randn(1, 1, n_mels, n_mels)).shape[1]

        self.gem = GeM1d(p=3.0, kernel_size=3)
        self.fc1 = nn.Linear(n_feat, n_feat, bias=True)
        self.att_block = AttBlock(n_feat, num_classes)
        self.dropout = nn.Dropout(head_dropout)
        self.distill_head = DistillHead(n_feat, 1536)

        init_layer(self.fc1)

    def forward(self, x):
        x = x.transpose(1, 2) 
        x = self.bn0(x)
        x = x.transpose(1, 2) 

        feat = self.backbone(x)
        feat = feat.mean(dim=2)
        
        feat = self.gem(feat)
        feat = self.dropout(feat)
        feat = feat.transpose(1, 2)
        feat = nn.functional.relu(self.fc1(feat))
        feat = feat.transpose(1, 2)
        feat = self.dropout(feat)

        clipwise = self.att_block(feat)
        return clipwise

class LogMelSpectrogramTransform(nn.Module):
    def __init__(self, mel_params: dict, top_db: float):
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(**mel_params)
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=top_db)
        self.top_db = top_db

    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        mel = self.mel_transform(wave)
        lms = self.db(mel)
        
        lms = torch.clamp((lms + self.top_db) / self.top_db, 0.0, 1.0)
        return lms[:, None, :, :]
        
print('Done')

In [ ]:
import tempfile
def ov_model_from_pt(pt_path, device="CPU", input_shape=(1, 1, 128, 313)):   
    model = BirdModel(
        model_name="tf_efficientnetv2_s.in21k",
        pretrained=False,
        drop_path_rate=0.15,
        drop_rate=0.2,
        num_classes=N_CLASSES,
        head_dropout=0.35,
        n_mels=128,
    )
    m, u = model.load_state_dict(torch.load(pt_path, map_location="cpu"), strict=False)
    print(f"missing={m}")
    print(f"unexpected={u}")
    model.eval()
    
    class InferenceWrapper(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
            
        def forward(self, x):
            clipwise = self.base_model(x)
            blended_logits = clipwise
            
            return blended_logits

    export_model = InferenceWrapper(model)
    export_model.eval()

    dummy = torch.randn(*input_shape, dtype=torch.float32)
    with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
        onnx_path = f.name

    try:
        torch.onnx.export(
            export_model, dummy, onnx_path,
            opset_version=17,
            do_constant_folding=True,
            input_names=["input"],
            output_names=["output"],
            dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}}
        )
        
        ov_model = ov.convert_model(onnx_path)
    finally:
        os.unlink(onnx_path)  

    compiled = ov.compile_model(
        ov_model, device,
        {
            "PERFORMANCE_HINT": "THROUGHPUT",
            "INFERENCE_NUM_THREADS": 4,
            "NUM_STREAMS": 2,
        }
    )
    return compiled

def gaussian_smooth_by_record(preds, kernel=GAUSSIAN_KERNEL):
    kernel = np.asarray(kernel, dtype=np.float32)
    kernel = kernel / kernel.sum()
    pad = len(kernel) // 2

    padded = np.pad(preds, ((0,0),(0,0),(pad,pad),(0,0)), mode="edge")
    smoothed = np.zeros_like(preds)
    for i, w in enumerate(kernel):
        smoothed += w * padded[:, :, i:i + preds.shape[2], :]
    return smoothed

def rank_normalize(x):
    r_x = np.zeros_like(x)
    for i in range(x.shape[1]):
        r_x_i = pd.Series(x[:, i]).rank(method="max")
        r_x[:, i] = r_x_i / r_x_i.shape[0]
    return r_x

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

wave_list = []
shifted_wave_list =[]
sample_rate = 32_000
max_sec = 60
duration = sample_rate * max_sec
shift = int(sample_rate * 2.5)

for path in tqdm(test_ss_paths, desc="Loading Audio"):
    with soundfile.SoundFile(path) as f:
        n_frames = f.frames
        wave = np.zeros(shift + duration + shift, dtype="float32")
        
        read_frames = duration + shift if n_frames >= duration + shift else -1
        audio_data = f.read(frames=read_frames, dtype="float32")
        
        if audio_data.ndim > 1:
            audio_data = audio_data.mean(axis=1)
            
        wave[shift:shift + len(audio_data)] = audio_data

    for seg_sec in range(0, max_sec, 5):
        wave_list.append(
            wave[shift + seg_sec * sample_rate: shift + (seg_sec + 5) * sample_rate])
    for seg_sec in range(0, max_sec + 5, 5):
        shifted_wave_list.append(
            wave[seg_sec * sample_rate: (seg_sec + 5) * sample_rate])
        
num_test_ss_audios = len(test_ss_paths)
num_test_ss_segs = len(wave_list)
num_shifted_test_ss_segs = len(shifted_wave_list)

if USE_PRIOR:
    test_sites = []
    test_hours =[]
    
    for p in test_ss_paths:
        site, hour = parse_fname(p.name)
        test_sites.append(site)
        test_hours.append(hour)
    
    normal_sites = np.repeat(test_sites, 12)
    normal_hours = np.repeat(test_hours, 12)
    
    shifted_sites = np.repeat(test_sites, 13)
    shifted_hours = np.repeat(test_hours, 13)
    
    with open("/kaggle/input/datasets/jakkma/prior-table/train_ss_prior_tables.pkl", "rb") as f:
        prior_tables = pickle.load(f)

print(f"Normal segments: {num_test_ss_segs} ({num_test_ss_segs / 12} files)")
print(f"Shifted segments: {num_shifted_test_ss_segs} ({num_shifted_test_ss_segs / 13} files)")

batch_size = 12
wave_batches =[]
for i in trange(0, len(wave_list), batch_size, desc="Stack Normal"):
    wave_batches.append(np.stack(wave_list[i: i + batch_size], axis=0))
del wave_list 

shifted_wave_batches =[]
for i in trange(0, len(shifted_wave_list), batch_size, desc="Stack Shifted"):
    shifted_wave_batches.append(np.stack(shifted_wave_list[i: i + batch_size], axis=0))
del shifted_wave_list 
gc.collect()

ov_engines =[]
for fold_id in range(N_FOLDS):
    pt_path = TRAINED_MODEL / f"best_model_fold{fold_id}.pt"
    engine = ov_model_from_pt(pt_path, device="CPU")
    ov_engines.append(engine)

test_preds_arr = np.zeros((N_FOLDS, num_test_ss_segs, N_CLASSES), dtype=np.float32)
shifted_test_preds_arr = np.zeros((N_FOLDS, num_shifted_test_ss_segs, N_CLASSES), dtype=np.float32)

lms_transform = LogMelSpectrogramTransform(mel_spectrogram_params, 
                                           top_db=top_db).eval()

for fold_id, engine in enumerate(ov_engines):
    print(f"[fold: {fold_id}] Normal Inference...")
    infer_queue = ov.AsyncInferQueue(engine, 16)
    logit_arr = np.zeros((num_test_ss_segs, N_CLASSES), dtype=np.float32)
    
    def callback_normal(request, userdata):
        logit_arr[userdata] = request.get_output_tensor().data

    infer_queue.set_callback(callback_normal)
    input_name = engine.inputs[0].get_any_name()

    total = 0
    for waves in tqdm(wave_batches, desc="Normal stream"):
        b_size = len(waves)
        idxs = np.arange(total, total + b_size)
        total += b_size
        
        lms = lms_transform(torch.from_numpy(waves)).numpy().astype(np.float32)
        infer_queue.start_async({input_name: lms}, userdata=idxs)

    infer_queue.wait_all() 
    
    if USE_PRIOR:
        logit_arr = apply_prior(logit_arr, normal_sites, normal_hours, prior_tables, lambda_prior=0.4)
    if USE_CLASS_TEMP:
        logit_arr = logit_arr / temperatures[None, :]
        
    test_preds_arr[fold_id] = sigmoid(logit_arr)

    print(f"[fold: {fold_id}] Shifted Inference...")
    infer_queue_shift = ov.AsyncInferQueue(engine, 16)
    shifted_logit_arr = np.zeros((num_shifted_test_ss_segs, N_CLASSES), dtype=np.float32)
    
    def callback_shift(request, userdata):
        shifted_logit_arr[userdata] = request.get_output_tensor().data

    infer_queue_shift.set_callback(callback_shift)

    total = 0
    for waves in tqdm(shifted_wave_batches, desc="Shifted stream"):
        b_size = len(waves)
        idxs = np.arange(total, total + b_size)
        total += b_size
        
        lms = lms_transform(torch.from_numpy(waves)).numpy().astype(np.float32)
        infer_queue_shift.start_async({input_name: lms}, userdata=idxs)

    infer_queue_shift.wait_all()
    
    if USE_PRIOR:
        shifted_logit_arr = apply_prior(shifted_logit_arr, shifted_sites, shifted_hours, prior_tables, lambda_prior=0.4)
    if USE_CLASS_TEMP:
        shifted_logit_arr = shifted_logit_arr / temperatures[None, :]
        
    shifted_test_preds_arr[fold_id] = sigmoid(shifted_logit_arr)

print(test_preds_arr.shape)

del wave_batches, shifted_wave_batches
gc.collect()
print('Done')

In [ ]:
test_preds_arr_by_record = test_preds_arr.reshape(N_FOLDS, num_test_ss_audios, 12, N_CLASSES)
shifted_test_preds_arr_by_record = shifted_test_preds_arr.reshape(
    N_FOLDS, num_test_ss_audios, 13, N_CLASSES)

test_preds_arr_tta_by_records = (
        0.25 * shifted_test_preds_arr_by_record[..., 0:12, :] +
        0.50 * test_preds_arr_by_record +
        0.25 * shifted_test_preds_arr_by_record[..., 1:13, :])

if USE_GAUSSIAN_SMOOTH:
    test_preds_arr_tta_by_records = gaussian_smooth_by_record(test_preds_arr_tta_by_records, GAUSSIAN_KERNEL)

test_preds_arr_tta = test_preds_arr_tta_by_records.reshape(N_FOLDS, num_test_ss_segs, N_CLASSES)
test_preds_arr_tta_rank = np.zeros_like(test_preds_arr_tta)


if RANK_AVG:
    for fold_id in range(N_FOLDS):
        test_preds_arr_tta_rank[fold_id] = rank_normalize(test_preds_arr_tta[fold_id])
    test_pred_avg = test_preds_arr_tta_rank.mean(axis=0)
else:
    test_pred_avg = test_preds_arr_tta.mean(axis=0)
    
if USE_FILE_SCALE:
    test_pred_avg = file_confidence_scale(test_pred_avg, n_windows=N_WINDOWS, top_k=2, power=0.4)
if USE_RANK_SCALE:
    test_pred_avg = rank_aware_scaling(test_pred_avg, n_windows=N_WINDOWS, power=0.4)
if USE_ADAPT_SMOOTH:
    test_pred_avg = adaptive_delta_smooth(test_pred_avg, n_windows=N_WINDOWS, base_alpha=0.20)
    
test_pred_avg = np.clip(test_pred_avg, 0.00, 1.00)

sub_df = pd.DataFrame(test_pred_avg, columns=CLASSES, 
                      index=pd.Series(test_ss_segs, name="row_id")).reset_index()

display(sub_df.head())
sub_df = pd.merge(sample_sub[["row_id"]], sub_df, on="row_id", how="left").fillna(0.0)
sub_df.to_csv("submission.csv", index=False)
print(sub_df.shape)
display(sub_df.head())